# Student Performance Analysis: Classification & Clustering
## IT326 — Final Notebook (Phase 1 + Phase 2 + Phase 3)

This notebook preserves the original work across all three phases, organized under the eight required final-report sections.

**GitHub file paths used:** `Dataset/raw_dataset.xlsx` and `Dataset/Preprocessed_dataset.csv`. Run from the repository root (or open from GitHub in Colab and ensure the Dataset folder is available in the working directory).


# 1. Problem

Student performance is influenced by academic, lifestyle, and background factors. This project investigates whether final grades can be predicted from student attributes and whether students form meaningful groups. The motivation is to understand patterns that may inform academic support.

## 1. Project Goal

Student performance is influenced by different academic, demographic, lifestyle, and socioeconomic factors. This project uses data from **750 students** with **15 features** covering these different aspects. Analyzing this data can help identify patterns and factors associated with different levels of academic performance.

We are tackling two tasks with this dataset:

- **Classification:** Predicting a student's final grade using academic, personal, and behavioral factors. The `final_grade` column is used as the class label, with possible categories **A, B, C, D, and F**. Features such as study hours, attendance rate, assignment score, midterm score, internet access, parent education, sleep hours, stress level, part-time job status, and extracurricular activities are used for the prediction.

- **Clustering:** Grouping students based on similar personal, academic, and behavioral characteristics (e.g., age, study hours, attendance rate, internet access, sleep hours, part-time job status, stress level, extracurricular activities). Grades and scores are **not** used to create the clusters. The goal is to identify natural groups of students with similar characteristics and then examine whether these groups show differences in academic performance.

### Motivation
Understanding the factors linked to student performance can provide useful insights and help identify students who may need additional academic support. This dataset allows us to apply classification to predict a student's final grade and clustering to discover groups of students with similar characteristics and patterns.


# 2. Data Mining Task

**Classification:** Predict `final_grade` (A, B, C, D, F) with Decision Trees using Gini and Information Gain (entropy).

**Clustering:** Apply K-Means to group students based on their selected numeric/transformed attributes without using the class label.

# 3. Data

The following cells retain the Phase 1 dataset description and the Phase 2 exploratory data analysis.

## 2. Dataset Source

- **Dataset name:** Student Performance Dataset
- **File:** [raw_dataset.xlsx](Dataset/raw_dataset.xlsx)
- **Source / URL:** https://www.kaggle.com/datasets/krisnadwikialdi/student-performance-dataset


## 3. Dataset Description

In this section we load the dataset and explore:
- Number and data types of attributes (features)
- Number of objects (instances/records)
- The class attribute (`final_grade`) and the count of instances per label
- A sample of the raw dataset


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_excel('Dataset/raw_dataset.xlsx')
df.head()

### 3.1 Number and Data Types of Attributes

In [ ]:
print(f"Number of attributes (columns): {df.shape[1]}")
print("\nData types of each attribute:")
df.dtypes

### 3.2 Number of Objects (Instances/Records)

In [ ]:
print(f"Number of instances (rows): {df.shape[0]}")

### 3.3 Class Attribute and Count of Instances per Label

In [ ]:
class_attribute = "final_grade"
print(f"Class attribute: {class_attribute}")
print(f"Possible values: {sorted(df[class_attribute].unique())}")
print("\nCount of instances per label:")
df[class_attribute].value_counts()

### 3.4 Sample of the Raw Dataset

In [ ]:
df.sample(10, random_state=42)

## 3.1 Data Summarization and Visualization (Phase 2)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set_style("whitegrid")


raw_df = pd.read_excel("Dataset/raw_dataset.xlsx")


df = raw_df.copy()

numeric_cols = ["age", "study_hours", "attendance_rate", "assignment_score",
                "midterm_score", "final_score", "sleep_hours", "stress_level"]
categorical_cols = ["gender", "internet_access", "parent_education",
                    "part_time_job", "extracurricular"]
target = "final_grade"

print("Shape:", raw_df.shape)
raw_df.head()

# 1. Data Analysis

## 1.1 Statistical Summaries

In [ ]:
raw_df.info()
print()
raw_df.describe().T

**Description:** `info()` shows the data types and `describe()` gives count, mean, std, min, quartiles and max for every numeric attribute.

**What it tells us:** Several values are impossible for this kind of data: `study_hours` has a negative minimum (-1.35), and `assignment_score`, `midterm_score` and `final_score` have maximums above 100 (107.6, 103.0 and 126.9). Also the attributes are on different scales (`attendance_rate` around 60-100, `stress_level` 1-9, `study_hours` around 0-9).  

**Decision:** The data needs noise/outlier treatment and normalization.

## 1.2 Missing Values Analysis

In [ ]:
missing = pd.DataFrame({
    "missing_count": raw_df.isnull().sum(),
    "missing_percent": (raw_df.isnull().sum() / len(raw_df) * 100).round(2)
})
print(missing)
print()
print("Duplicated rows:", raw_df.duplicated().sum())

**Description:** Count and percentage of missing values per attribute, plus a check for duplicated rows.  
**What it tells us:** There are no missing values in any attribute and no duplicated rows.  
**Decision:** No imputation or row removal is needed.

## 1.3 Five Number Summary and Outliers (IQR method)

In [ ]:
rows = []
for col in numeric_cols:
    q1 = raw_df[col].quantile(0.25)
    q3 = raw_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((raw_df[col] < lower) | (raw_df[col] > upper)).sum()
    rows.append([col, raw_df[col].min(), q1, raw_df[col].median(), q3,
                 raw_df[col].max(), round(lower, 2), round(upper, 2), n_out])

summary = pd.DataFrame(rows, columns=["attribute", "min", "Q1", "median", "Q3",
                                      "max", "lower_bound", "upper_bound", "outliers"])
summary

**Description:** The five-number summary shows the minimum, first quartile (Q1), median, third quartile (Q3), and maximum for each numerical attribute. The IQR method was also used to identify potential outliers.

**What it tells us:** Potential outliers were detected in `study_hours`, `assignment_score`, `midterm_score`, `final_score`, and `sleep_hours`. However, not all statistical outliers are necessarily invalid. Negative `study_hours` values and scores above 100 are domain-invalid, while extreme `sleep_hours` values may still be logically possible.

**Decision:** Preprocessing is required only for domain-invalid values, such as negative `study_hours` and scores above 100. Statistically extreme but logically valid values can be retained.

## 1.4 Boxplots (numeric attributes)

In [ ]:
plot_cols = [
    "age", 
    "study_hours",
    "attendance_rate",
    "assignment_score",
    "midterm_score",
    "final_score",
    "sleep_hours",
    "stress_level"
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ax, col in zip(axes.flatten(), plot_cols):
    sns.boxplot(y=raw_df[col], ax=ax, color="skyblue")
    ax.set_title("Boxplot of " + col)

plt.tight_layout()
plt.show()

**Description:** Boxplots show the distribution of each numerical attribute using the median, quartiles, spread, and potential outliers.

**What it tells us:** The boxplots reveal potential outliers in several numerical attributes. Some values are domain-invalid, such as negative `study_hours` and scores above 100, while other extreme values may still be logically valid.

**Decision:** Domain-invalid values should be corrected using valid boundaries, while statistically extreme but logically valid values can be retained.

## 1.5 Histograms (variable distributions of numeric attributes)

In [ ]:
hist_cols = ["age", "study_hours", "attendance_rate", "assignment_score",
             "midterm_score", "final_score", "sleep_hours", "stress_level"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), hist_cols):
    sns.histplot(raw_df[col], bins=20, kde=True, ax=ax, color="steelblue")
    ax.set_title("Distribution of " + col)
plt.tight_layout()
plt.show()

print(raw_df[hist_cols].skew().round(2))

**Description:** Histograms show the distribution of each numerical attribute, while the density curve and skewness values help describe the shape and symmetry of the distributions.

**What it tells us:** Most numerical attributes are approximately symmetric, with skewness values close to zero, so no skewness correction is needed. However, the numerical attributes have different ranges, and some score attributes contain extreme values beyond their valid limits.

**Decision:** Normalization is needed for selected numerical attributes with different ranges to place them on a common scale, while domain-invalid values are handled separately during noise removal.

## 1.6 Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=raw_df, x="study_hours", y="final_score", hue=target, ax=axes[0])
axes[0].set_title("study_hours vs final_score")
sns.scatterplot(data=raw_df, x="midterm_score", y="final_score", hue=target, ax=axes[1])
axes[1].set_title("midterm_score vs final_score")
sns.scatterplot(data=raw_df, x="attendance_rate", y="assignment_score", hue=target, ax=axes[2])
axes[2].set_title("attendance_rate vs assignment_score")
plt.tight_layout()
plt.show()

**Description:** Scatter plots of pairs of numeric attributes, coloured by the class label.  
**What it tells us:** The points are spread without a clear pattern (no strong linear relationship between the pairs), and the grade colours are mixed everywhere. The extreme points (negative `study_hours`, `final_score` above 100) are clearly separated from the main cloud.  
**Decision:** Confirms that the extreme points are noise that should be treated.

## 1.7 Bar Plots (nominal attributes)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.flatten(), categorical_cols):
    sns.countplot(data=raw_df, x=col, ax=ax, color="salmon")
    ax.set_title("Bar plot of " + col)
axes[1][2].axis("off")
plt.tight_layout()
plt.show()

for col in categorical_cols:
    print(raw_df[col].value_counts())
    print()

**Description:** Bar plots of the count of each category in the nominal attributes.  
**What it tells us:** The categories are fairly balanced, except `internet_access` (632 Yes vs 118 No). `gender`, `internet_access`, `part_time_job` and `extracurricular` have two values, and `parent_education` has four values (High School, Bachelor, Master, PhD) with a natural order. These attributes are stored as text.  
**Decision:** Categorical variable transformation is needed because the algorithms require numeric input.

## 1.8 Class Label Distribution

In [ ]:
order = ["A", "B", "C", "D", "F"]
plt.figure(figsize=(7, 5))
sns.countplot(data=raw_df, x=target, order=order, color="mediumseagreen")
plt.title("Class label distribution (final_grade)")
plt.show()

print(raw_df[target].value_counts().reindex(order))
print()
print((raw_df[target].value_counts(normalize=True).reindex(order) * 100).round(1))

**Description:** Number of records in each class of `final_grade`.  
**What it tells us:** The five classes are almost balanced (A 150, B 142, C 143, D 154, F 161), so there is no class imbalance problem.  
**Decision:** No resampling is needed, and the original class distribution is preserved.

# 4. Data Preprocessing

The following original Phase 2 code applies noise removal, discretization, categorical transformation, and normalization. It saves a separate preprocessed CSV in the existing GitHub Dataset folder.

# 2. Data Preprocessing

According to the phase guidelines, we apply appropriate preprocessing tasks chosen from the recommended techniques: **Noise Removal**, **Discretization**, **Normalization** and **Variable Transformation (Categorical)**

## 2.1 Noise Removal

**Justification:** The attributes `study_hours`, `assignment_score`, `midterm_score`, and `final_score` contain domain-invalid values. Negative study hours and scores above 100 are not logically valid and may result from data entry errors or uncalibrated extra credit.

**Method:** Domain boundary clamping was applied by setting the lower threshold of `study_hours` to 0.0 and capping `assignment_score`, `midterm_score`, and `final_score` at 100.0.

**Improvement:** This corrects invalid values while preserving the relative ordering of students, avoiding unnecessary removal of records or replacement of valid extreme values.

In [ ]:
print("--- Preprocessing: Noise Removal (Domain Boundary Clamping) ---")

df["study_hours"] = df["study_hours"].clip(lower=0.0)
for col in ["assignment_score", "midterm_score", "final_score"]:
  df[col] = df[col].clip(upper=100.0)

print("Updated ranges after domain boundary clamping:")
df[["study_hours", "assignment_score", "midterm_score", "final_score"]].describe().T[["min", "mean", "max"]]

## 2.2 Discretization

**Justification:** `age` and `study_hours` were selected because they are numerical attributes that can be meaningfully grouped into categories, making the data easier to interpret and analyze.

**Method:** Discretization was applied using `pd.cut()` with three bins. `age` was divided into Young, Middle, and Older, while `study_hours` was divided into Low, Medium, and High.

**Improvement:** This transformation groups numerical values into categorical intervals, simplifying the data representation and making patterns in `age` and `study_hours` easier to analyze.

In [ ]:
column_to_discretize = "age"
num_bins = 3
df["discretized_age"] = pd.cut(
    df[column_to_discretize],
    bins=num_bins,
    labels=["Young", "Middle", "Older"])
print("Original DataFrame:")
display(df[["age", "discretized_age"]].head(15))

In [ ]:
column_to_discretize = "study_hours"
num_bins = 3
df["discretized_study_hours"] = pd.cut(
    df[column_to_discretize],
    bins=num_bins,
    labels=["Low", "Medium", "High"])
print("Original DataFrame:")
display(df[["study_hours", "discretized_study_hours"]].head(15))

## 2.3 Variable Transformation (Categorical)

**Justification:** Machine learning algorithms require numerical inputs and cannot directly perform mathematical operations on textual categorical data such as `gender`, `internet_access`, `parent_education`, `part_time_job`, `extracurricular`, and `final_grade`.

**Method:** Binary mapping (0/1) was applied to the two-category nominal attributes `gender`, `internet_access`, `part_time_job`, and `extracurricular`. Ordinal mapping was applied to `parent_education` (0 to 3) and `final_grade` (0 to 4) to preserve their natural order.

**Improvement:** This transformation converts categorical attributes into numerical formats compatible with data mining methods while preserving the meaningful order of education levels and letter grades without adding unnecessary columns.

In [ ]:
# 1. Binary nominal 
df['gender_transformed'] = df['gender'].replace({'Female': 1, 'Male': 0})
df['internet_access_transformed'] = df['internet_access'].replace({'Yes': 1, 'No': 0})
df['part_time_job_transformed'] = df['part_time_job'].replace({'Yes': 1, 'No': 0})
df['extracurricular_transformed'] = df['extracurricular'].replace({'Yes': 1, 'No': 0})
# 2. Ordinal variable
df['parent_education_transformed'] = df['parent_education'].replace({'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3})
# 3. Target class label mapped to standard academic grade order
df['final_grade_transformed'] = df['final_grade'].replace({'F': 0, 'D': 1, 'C': 2, 'B': 3, 'A': 4})

display(df[[
    'gender', 'gender_transformed',
    'parent_education', 'parent_education_transformed',
    'internet_access','internet_access_transformed',
    'part_time_job', 'part_time_job_transformed' ,
    'extracurricular' , 'extracurricular_transformed',
    'final_grade', 'final_grade_transformed'
]].head(5))

## 2.4 Normalization

**Justification:** The selected numerical attributes (`attendance_rate`, `assignment_score`, `midterm_score`, `final_score`, and `stress_level`) have different numerical ranges. Normalization is applied to place these attributes on a common scale and prevent attributes with larger ranges from having a greater influence on distance-based analysis.

**Method:** Min-Max normalization was applied to `attendance_rate`, `assignment_score`, `midterm_score`, `final_score`, and `stress_level`, transforming their values to the range [0, 1]. The attributes `student_id`, `age`, `study_hours`, and `sleep_hours` were excluded from normalization, along with categorical attributes and the target variable `final_grade`.

**Improvement:** The selected numerical attributes are now represented on the same scale, reducing the effect of differences in their original ranges. This improves their suitability for distance-based data mining methods such as clustering.

In [ ]:
# Numerical attributes selected for normalization
cols_to_normalize = [
    "attendance_rate",
    "assignment_score",
    "midterm_score",
    "final_score",
    "stress_level"
]

# Check ranges before normalization
print("Numeric attribute ranges before normalization:")
df[cols_to_normalize].agg(["min", "max"]).T

In [ ]:
# Apply Min-Max normalization
scaler = MinMaxScaler()

df[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])

print("Ranges after Min-Max normalization:")
df[cols_to_normalize].agg(["min", "max"]).T

In [ ]:
# Check that the raw data was not modified
raw_check = pd.read_excel("Dataset/raw_dataset.xlsx")
print("Raw data unchanged:", raw_df.equals(raw_check))

print("Preprocessed Dataset:")
display(df.head())

df.to_csv("Dataset/Preprocessed_dataset.csv", index=False)

print("Preprocessed dataset saved successfully!")
print("Final shape:", df.shape)

# 5. Data Mining Technique

**Decision Tree:** `sklearn.tree.DecisionTreeClassifier`, criteria `gini` and `entropy`, with 70/30, 80/20 and 90/10 train/test partitions.

**K-Means:** `sklearn.cluster.KMeans` with K = 2, 3 and 19; compare the Elbow WSS and Silhouette scores. PCA is used only for two-dimensional visualization.

# 6. Evaluation and Comparison

## 6.1 Classification — Decision Tree

The following cells retain the Phase 3 classification experiments, comparison table, confusion matrices, and decision-tree visualization.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn import tree
import matplotlib.pyplot as plt

In [ ]:
# Load the preprocessed student dataset
df = pd.read_csv("Dataset/Preprocessed_dataset.csv")
print(df.head())

# Use final_grade as the target
y = df['final_grade']

# Remove the ID, target, and columns we do not need
X = df.drop(columns=[
    'student_id',
    'final_grade',
    'final_grade_transformed',
    'final_score',
    'gender',
    'internet_access',
    'parent_education',
    'part_time_job',
    'extracurricular'
])

# Convert the remaining categorical columns into numbers
X = pd.get_dummies(X)

fn = X.columns.tolist()

print("Features:", fn)
print("Dataset shape:", X.shape)

In [ ]:

# Try three different training and testing splits
partitions = {
    "70/30": 0.3,
    "80/20": 0.2,
    "90/10": 0.1
}

In [ ]:

# Compare Gini and Entropy to see which one works better
criteria = ['gini', 'entropy']

results = []
result_matrices = []
best_accuracy = -1


In [ ]:

# Train and test the Decision Tree using different splits
for partition, test_size in partitions.items():

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=1
    )

    # Try Gini and Entropy for each split
    for criterion in criteria:

        clf = DecisionTreeClassifier(
            criterion=criterion,
            random_state=1
        )

        clf = clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # Check how well the model performs
        accuracy = metrics.accuracy_score(y_test, y_pred)

        train_accuracy = metrics.accuracy_score(
            y_train, clf.predict(X_train)
        )

        f1 = metrics.f1_score(
            y_test, y_pred,
            average='weighted',
            zero_division=0
        )

        cm = confusion_matrix(
            y_test, y_pred,
            labels=clf.classes_
        )

        print("\nPartition:", partition)
        print("Criterion:", criterion)
        print("Training Accuracy:", train_accuracy)
        print("Testing Accuracy:", accuracy)
        print("F1-score:", f1)
        print("Confusion Matrix:")
        print(cm)

        # Save the results to compare them later
        results.append({
            "Partition": partition,
            "Criterion": criterion,
            "Training Accuracy": train_accuracy,
            "Accuracy": accuracy,
            "F1-score": f1
        })

        # Save each confusion matrix for the plots
        result_matrices.append({
            "Partition": partition,
            "Criterion": criterion,
            "Matrix": cm,
            "Labels": clf.classes_
        })

        # Keep the model with the highest testing accuracy
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_clf = clf
            best_partition = partition
            best_criterion = criterion


In [ ]:

# Put all six results in one table
results_df = pd.DataFrame(results)

print("Comparison Results:")
print(results_df.to_string(index=False))

# Show the best result from our experiments
print("\nBest Partition:", best_partition)
print("Best Criterion:", best_criterion)
print("Best Accuracy:", best_accuracy)


In [ ]:

# Compare the accuracy of Gini and Entropy
accuracy_table = results_df.pivot(
    index="Partition",
    columns="Criterion",
    values="Accuracy"
)

accuracy_table = accuracy_table.reindex(
    ["70/30", "80/20", "90/10"]
)

accuracy_table.plot(kind="bar", figsize=(8, 5))

plt.title("Decision Tree Accuracy Comparison")
plt.xlabel("Data Partition")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Criterion")
plt.tight_layout()
plt.show()


In [ ]:

# Draw the tree to see how the model makes its decisions
plt.figure(figsize=(16, 8))

tree.plot_tree(
    best_clf,
    feature_names=fn,
    class_names=[str(c) for c in best_clf.classes_],
    filled=True,
    max_depth=3,
    fontsize=8
)

plt.title("Best Decision Tree")
plt.show()


In [ ]:

# Show the confusion matrix for all six experiments
fig, axes = plt.subplots(3, 2, figsize=(12, 14))

for ax, result in zip(axes.flat, result_matrices):

    disp = metrics.ConfusionMatrixDisplay(
        confusion_matrix=result["Matrix"],
        display_labels=result["Labels"]
    )

    disp.plot(
        ax=ax,
        cmap="Blues",
        colorbar=False
    )

    ax.set_title(
        result["Partition"] + " - " + result["Criterion"]
    )

plt.tight_layout()
plt.show()


**Description:** The table compares the performance of Decision Tree Classification using Gini and Entropy with three different data partitions (70/30, 80/20, and 90/10).

**What it tells us:** The results show differences in accuracy between Gini and Entropy across the three partitions. The training accuracy is also compared with the testing accuracy to check for overfitting.

**Conclusion:** The best-performing configuration is identified based on testing accuracy. However, the difference between training and testing accuracy should also be considered when evaluating the model.

## 6.2 Clustering — K-Means

The following cells retain the Phase 3 clustering experiments, selection of K, evaluation plots, and comparisons.

# 2. Clustering

## 2.1 Preparing the Data for Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load
df = pd.read_csv("Dataset/Preprocessed_dataset.csv")

# Keep the class label aside
target = df["final_grade"]

# Remove the ID, the class label, and the text columns
# (the text columns already have numeric versions ending in _transformed)
# We also remove gender, because we want clusters based on student behaviour and performance
features = df.drop(columns=[
    "student_id",
    "final_grade",
    "final_grade_transformed",
    "gender",
    "gender_transformed",
    "internet_access",
    "parent_education",
    "part_time_job",
    "extracurricular",
    "discretized_age",
    "discretized_study_hours"
])

# Scale the features so no attribute dominates the distance
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
SG_scaled = pd.DataFrame(scaled_features, columns=features.columns)

print("Features:", SG_scaled.columns.tolist())
print("Dataset shape:", SG_scaled.shape)
print("\nScaled DataFrame:")
print(SG_scaled.head())

## 2.2 Choosing the Number of Clusters (k)

### 2.2.1 The emperical rule

In [ ]:
# Empirical rule: k= the square root of n/2
n = len(SG_scaled)
empirical_k = np.sqrt(n / 2)

print("Number of objects (n):", n)
print("sqrt(n/2) =", round(empirical_k, 2))
print("Chosen k from the empirical rule:", round(empirical_k))

### 2.2.2 The Elbow method (WSS only)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Elbow method: run k-means for k = 1 to 20 and record the WSS
k_values = list(range(1, 21))
wss_values = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=25)
    kmeans.fit(SG_scaled)
    wss_values.append(kmeans.inertia_)

# Plot the elbow curve (WSS only)
plt.figure(figsize=(8, 5))
plt.plot(k_values, wss_values, marker='o', linestyle='-', color='blue')
plt.axvline(x=3, linestyle='--', color='red', label='Elbow: k = 3')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-cluster Sum of Squares (WSS)')
plt.title('Elbow Method for Optimal k (1 to 20)')
plt.xticks(k_values)
plt.legend()
plt.grid(True)
plt.show()

# Show how much the WSS drops each time we add a cluster
for i in range(1, 10):
    print(f"k = {k_values[i-1]} -> {k_values[i]}: WSS drops by {wss_values[i-1] - wss_values[i]:.1f}")

### 2.2.3 The Sihlouette method (average silhouette only)

In [ ]:
from sklearn.metrics import silhouette_score

# Silhouette method: run k-means for k = 2 to 20 and record the average silhouette
sil_k_values = list(range(2, 21))
sil_values = []

for k in sil_k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=25)
    labels = kmeans.fit_predict(SG_scaled)
    sil_values.append(silhouette_score(SG_scaled, labels))

# Best k = the one that maximizes the average silhouette
best_k = sil_k_values[int(np.argmax(sil_values))]

# Plot the silhouette curve
plt.figure(figsize=(8, 5))
plt.plot(sil_k_values, sil_values, marker='o', linestyle='-', color='purple')
plt.axvline(x=best_k, linestyle='--', color='green',
            label=f'Max silhouette: k = {best_k}')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Average Silhouette Coefficient')
plt.title('Silhouette Method for Optimal k (2 to 20)')
plt.xticks(sil_k_values)
plt.legend()
plt.grid(True)
plt.show()

# Print the values
for k, s in zip(sil_k_values, sil_values):
    print(f"k = {k}: average silhouette = {s:.4f}")
print(f"\nBest k by silhouette: {best_k}")

**Description:** We used three methods from the lecture to choose the number of clusters.
The empirical rule gives k = √(750/2) ≈ 19.36, so k = 19.
The elbow method plots the total within-cluster sum of squares (WSS) for k = 1 to 20.
The silhouette method gives the highest average silhouette at k = 2.

**What it tells us:** The WSS always decreases as k increases. The largest drops are from k = 1 to 2 (765) and from k = 2 to 3 (645). After k = 3 the drop is less than half as large (298), and the curve keeps declining gradually. So the elbow is at k = 3. The elbow is not very sharp, which suggests that the students do not form strongly separated groups.

**Conclusion:** We selected three values of k: **k = 19** (empirical rule), **k = 3** (elbow method), and **k = 2** (highest average silhouette).

## 2.3 K-Means Clustering

In [ ]:
# Set a fixed seed so the results are the same every time
np.random.seed(42)

# This function runs k-means for a given k and prints the results
def run_kmeans(k):

    # Run k-means clustering with k clusters
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=25)
    kmeans_result = kmeans.fit(SG_scaled)

    # Print the clustering result
    print(f"===== k = {k} =====")

    print("\nCluster Centers:")
    print(pd.DataFrame(kmeans_result.cluster_centers_, columns=SG_scaled.columns).round(2))

    print("\nCluster Labels (first 20):")
    print(kmeans_result.labels_[:20])

    print("\nObjects per cluster:")
    print(pd.Series(kmeans_result.labels_).value_counts().sort_index())

    return kmeans_result

**by Empirical Rule**

In [ ]:
# 1st call: k = 19 (by empirical rule)
kmeans_result_19= run_kmeans(19)

**by elbow method**

In [ ]:
# 2nd call: k = 3 (by elbow method)
kmeans_result_3= run_kmeans(3)

**By highest silhouette avg**

In [ ]:
# 3rd call: k = 2 (by highest silhouette)
kmeans_result_2= run_kmeans(2)

**Description:** We applied k-means clustering with k = 19, k = 3, and k = 2, using the same settings for each run (random_state = 42, n_init = 25). For each k we the cluster centers, and the number of students in each cluster. We clustered on 12 attributes: we excluded gender, because it is a demographic attribute and we wanted clusters based on student behaviour and performance.

**What it tells us:**
- **k = 19:** The clusters are small (21 to 64 students each) and the centers show no clear pattern.
- **k = 3:** Cluster 2 (118 students) contains students without internet access (internet_access_transformed = −2.31). Clusters 0 and 1 split the remaining students by extracurricular activities (0.98 vs −1.02). The academic attributes are close to 0 in all clusters.
- **k = 2:** The data is split almost entirely by internet access: 118 students without access and 632 with access.

**Conclusion:** The clusters are mainly formed by the binary attributes (internet access and extracurricular activities), not by academic performance. The empirical rule over-estimates k for a dataset of this size. The comparison of the three values of k is in Section 2.4.

## 2.4 Evaluation


### 2.4.1 Using Elbow Method

In [ ]:
# Elbow evaluation of the three chosen k values (2, 3, 19)
wss_2 = kmeans_result_2.inertia_
wss_3 = kmeans_result_3.inertia_
wss_19 = kmeans_result_19.inertia_
total_ss = wss_values[0]   # WSS at k = 1 = total variance of the data

elbow_table = pd.DataFrame({
    "k": [2, 3, 19],
    "WSS": [wss_2, wss_3, wss_19]
})
elbow_table["% reduction vs k = 1"] = 100 * (total_ss - elbow_table["WSS"]) / total_ss
print("WSS at k = 1:", round(total_ss, 1))
print(elbow_table.round(1).to_string(index=False))



**Conclusion:** The WSS at k = 1 is 9000 (the total variance of the 750 scaled students). The WSS of the three chosen values is:

| k | WSS | % reduction vs k = 1 |
|---|---|---|
| 2 | 8235.4 | 8.5% |
| 3 | 7590.5 | 15.7% |
| 19 | 5424.9 | 39.7% |

- **k = 3** is the turning point of the curve. Going from k = 2 to k = 3 still reduces the WSS by 645, but after k = 3 the gain per extra cluster falls below 300.
- **k = 2** is slightly too small, because adding one more cluster still gives a large reduction in WSS (645).
- **k = 19** has the lowest WSS, but this is expected since the WSS always decreases when k increases. From k = 3 to k = 19, 16 extra clusters reduce the WSS by only 2165.6 (about 135 per cluster), so the gain is small compared with the number of clusters added.
- The elbow is not sharp: even with 19 clusters the WSS is only 39.7% lower than with one cluster, so the students do not form very compact groups.

According to the Elbow method, **k = 3 is the best choice** among the three values.


### 2.4.2 Using Silhouette Coefficient

**Clusters Visualization**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pandas as pd

# 1. Apply PCA to reduce the features down to 2 dimensions using your scaled DataFrame
pca = PCA(n_components=2, random_state=42)
principal_components = pca.fit_transform(SG_scaled)

# Create a DataFrame for the PCA results
pca_df = pd.DataFrame(
    data=principal_components,
    columns=['Principal Component 1', 'Principal Component 2']
)

# 2. Loop through each required value of K (2, 3, and 19)
k_values = [2, 3, 19]

for k in k_values:
    # Run k-means clustering with the current K
    kmeans = {2: kmeans_result_2, 3: kmeans_result_3, 19: kmeans_result_19}[k]
    cluster_labels = kmeans.labels_

    # Assign cluster labels to the PCA DataFrame
    pca_df['Cluster'] = cluster_labels

    # 3. Visualize using a scatter plot colored by cluster label
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x='Principal Component 1',
        y='Principal Component 2',
        hue='Cluster',
        data=pca_df,
        palette='viridis',
        edgecolor='k',
        s=60,
        legend='full'
    )
    plt.title(f'K-Means Cluster Visualization (K = {k})')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

**Silhouette Plots**

In [ ]:
# Silhouette evaluation of the three chosen k values (2, 3, 19)
from sklearn.metrics import silhouette_samples
import matplotlib.cm as cm

models = {2: kmeans_result_2, 3: kmeans_result_3, 19: kmeans_result_19}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (k, model) in zip(axes, models.items()):
    labels = model.labels_
    sample_sil = silhouette_samples(SG_scaled, labels)   # silhouette of every student
    avg_sil = sample_sil.mean()
    print(f"k = {k}: average silhouette = {avg_sil:.3f}")

    y_lower = 10
    for c in range(k):
        c_sil = np.sort(sample_sil[labels == c])
        y_upper = y_lower + len(c_sil)
        color = cm.nipy_spectral(c / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,facecolor=color, edgecolor=color, alpha=0.7)
        y_lower = y_upper + 10

    ax.axvline(avg_sil, color='red', linestyle='--', label=f'Average = {avg_sil:.3f}')
    ax.set_xlim(-0.3, 0.6)
    ax.set_yticks([])
    ax.set_xlabel('Silhouette coefficient values')
    ax.set_ylabel('Cluster label')
    ax.set_title(f'Silhouette plot for k = {k}')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()


**Conclusion:** The silhouette coefficient measures how compact each cluster is and how well it is separated from the other clusters. Values close to 1 are preferable, and negative values mean that an object is closer to another cluster than to its own.

| k | Average silhouette |
|---|---|
| 2 | 0.145 |
| 3 | 0.098 |
| 19 | 0.081 |

- **k = 2** has the highest average silhouette of the three values (and of all values from 2 to 20).
- **k = 3** is lower, because the extra split divides the students with internet access by extracurricular activities only, so the two new clusters are close to each other.
- **k = 19** has the lowest value, because the 19 small clusters lie close to each other.
- All the values are low (the best is 0.145, far from 1), so the clusters are not strongly separated.

According to the silhouette coefficient, **k = 2 is the best choice** among the three values.


***Compare the results***

---



| k | Chosen by | WSS | Average silhouette | Cluster sizes |
|---|---|---|---|---|
| 2 | Highest silhouette | 8235.4 | **0.145** | 118 / 632 |
| 3 | Elbow method | 7590.5 | 0.098 | 118 / 312 / 320 |
| 19 | Empirical rule | **5424.9** | 0.081 | 21 to 64 |

- **Elbow method:** the WSS decreases as k increases, so k = 19 has the lowest WSS. The turning point of the curve is at k = 3, so k = 3 is the best of the three values.
- **Silhouette coefficient:** k = 2 has the highest average silhouette (0.145), and k = 19 the lowest (0.081), so k = 2 is the best of the three values.

**Agreement between the methods:**

 both methods reject k = 19. For k = 19 the WSS is lowest only because it has the largest number of clusters, and its silhouette is the lowest.
 The methods disagree between k = 2 and k = 3, because the Elbow method measures only compactness (and always rewards a larger k)while the silhouette measures compactness and separation together and penalizes the k = 3 split whose new clusters are close to each other.

**Interpretation:**

for k = 2, the students are split by internet access (118 without and 632 with).
For k = 3, the students with internet access are split further by extracurricular activities.
For k = 19, the clusters are small and show no clear pattern.
 In all cases the clusters are formed by these binary attributes and not by the academic attributes, and the low silhouette values show that the cluster structure is weak.

**Final conclusion:**

k = 19 (empirical rule) is not suitable for this dataset. k = 2 gives the best separation between clusters (highest silhouette), while k = 3 gives the best compactness-to-size balance (elbow).
We consider **k = 2** the best clustering because the silhouette coefficient evaluates both compactness and separation.


# 7. Findings and Discussion

## 7.1 Classification
The reported best model is the Decision Tree with **entropy** and the **80/20** partition (test accuracy **26.67%**). All six trees show **100% training accuracy** but much lower testing accuracy, suggesting severe overfitting. The prominence of `assignment_score` in the tree is a model-specific observation, **not evidence of causation**.

## 7.2 Clustering
The evaluated K values are **2, 3, and 19**. Reported average Silhouette scores are **0.145**, **0.098**, and **0.081**, respectively. K = 2 is preferred **based on the highest Silhouette**, although the Elbow interpretation favors K = 3; thus there is no literal majority vote selecting K = 2. The low Silhouette scores suggest weak cluster separation. The K = 2 grouping primarily reflects internet access rather than academic performance.

## 7.3 Overall Findings and Limitations
Neither method demonstrates strong predictive or naturally separated grade-related patterns in this dataset. The findings should be interpreted cautiously. Future work could include pruning/cross-validation and clustering methods more appropriate for mixed categorical and numerical data.

## 7.4 Comparison with Related Research
The final report compares these results with Hasan et al. [2], who studied Decision Trees and K-Means for student performance and reported much stronger predictive metrics. Different datasets, feature choices, and evaluation methods limit direct comparisons. Refer to the cited paper for its experimental context.


# 8. References

[1] krisnadwikialdi, “Student Performance Dataset,” Kaggle. [Online]. Available: https://www.kaggle.com/datasets/krisnadwikialdi/student-performance-dataset

[2] M. M. Hasan *et al*., “Predicting student performance and identifying learning behaviors using decision trees and K-means clustering,” *International Journal of Evaluation and Research in Education*, vol. 14, no. 5, 2025, doi: 10.11591/ijere.v14i5.33815.

[3] G. Feng, M. Fan, and Y. Chen, “Analysis and prediction of students’ academic performance based on educational data mining,” *IEEE Access*, vol. 10, 2022, doi: 10.1109/ACCESS.2022.3151652.

[4] J. Han, M. Kamber, and J. Pei, *Data Mining: Concepts and Techniques*, 3rd ed. Morgan Kaufmann, 2012.

[5] F. Pedregosa *et al*., “Scikit-learn: Machine learning in Python,” *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.
